# analyze_v4_no_csc2_candidates.ipynb

Downstream analysis for the **55 v4 sources WITHOUT a CSC2 match within 3σ_pos**.

Reads `outputs/v4_crossmatch_table.csv` and filters to
`has_csc2_match_within_3sigma == False`.

## Critical caveat — NOT X-ray non-detections

A row here means **no CSC2 source within 3σ_pos_heuristic of the SPT centroid**.
It does **NOT** mean Chandra has observed this position and found nothing:

| Case | Inferred? |
|---|---|
| Outside Chandra footprint | possible — `csc2_covered` is `Unknown` for all rows |
| Within footprint but below detection threshold | possible |
| CSC2 source > 3σ_pos away (genuine but missed by cut) | possible |
| CSC2 source > 3σ_pos due to large pointing error | possible |

`csc2_covered = 'Unknown'` will stay until a CSC2 footprint API is added
(see `scripts/build_v4_crossmatch_table.py`).
Without that, treating "no CSC2 match" as "no X-ray emission" overclaims.

## Scope of this notebook

1. Counterpart fingerprint per source — which catalogs matched, which didn't
2. \|b\| distribution + comparison to the 6 CSC2-matched sources
3. Multi-wavelength radio fluxes + simple spectral indices
4. AllWISE W1-W2 with Stern+2012 caveats
5. Top-N (|SNR|) per-source cutout panels — all 73 as PNGs come from
   `scripts/plot_source_panels.py --all`


In [ ]:
import os, sys
REPO = os.path.dirname(os.path.abspath('.'))   # notebooks/ -> repo root
sys.path.insert(0, os.path.join(REPO, 'src'))
from paths import OUT

# ── Load + filter to no-CSC2 subset ──
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

df_all = pd.read_csv(os.path.join(OUT, 'v4_crossmatch_table.csv'))
df = df_all[~df_all['has_csc2_match_within_3sigma']].reset_index(drop=True)

print(f'Loaded {len(df_all)} rows from v4_crossmatch_table.csv')
print(f'Filtered to {len(df)} sources with NO CSC2 match within 3σ_pos_heuristic')

# Sort by |SNR| descending to surface strongest sources first
df = df.sort_values('snr_max', key=abs, ascending=False).reset_index(drop=True)

print()
print('Top 15 by |SNR|:')
cols = ['id', 'snr_max', 'abs_b_deg', 'sigma_pos_arcsec_heuristic',
        'simbad_name', 'simbad_otype']
print(df[cols].head(15).to_string(index=False))


In [ ]:
# ── Counterpart fingerprint matrix ──
# One character per catalog: its initial if matched, '·' if not.

CATS = ['simbad', 'AT20G', 'NVSS', 'RACS-mid', 'VLASS', 'AllWISE']
matched = (df.reindex(columns=[f'{cat}_match' for cat in CATS], fill_value=False)
             .fillna(False).astype(bool))
df['counterpart_fingerprint'] = [
    ''.join(cat[0].upper() if m else '·' for cat, m in zip(CATS, row))
    for row in matched.itertuples(index=False)]


# Aggregate
fp_counts = df['counterpart_fingerprint'].value_counts()
print('Counterpart fingerprint counts (column order: simbad AT20G NVSS RACS-mid VLASS AllWISE):')
print('  legend: S=SIMBAD, A=AT20G, N=NVSS, R=RACS-mid, V=VLASS, A=AllWISE, ·=miss')
print()
print(fp_counts.to_string())

# Cross-tab: has-radio (any) vs has-IR (AllWISE) vs has-SIMBAD
df['has_any_radio'] = df[['AT20G_match','NVSS_match','RACS-mid_match','VLASS_match']].fillna(False).any(axis=1)
df['has_ir']       = df['AllWISE_match'].fillna(False).astype(bool)
df['has_simbad']   = df['simbad_match'].fillna(False).astype(bool)

print()
print('Three-way summary:')
ct = pd.crosstab([df['has_any_radio'], df['has_ir']], df['has_simbad'])
print(ct.to_string())


In [ ]:
# ── |b| distribution: no-CSC2 vs CSC2-matched ──
import matplotlib.pyplot as plt

df_csc2 = df_all[df_all['has_csc2_match_within_3sigma']]

fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(0, 7, 22)
ax.hist(df['abs_b_deg'],      bins=bins, alpha=0.55, color='steelblue',
        label=f'no CSC2 (n={len(df)})')
ax.hist(df_csc2['abs_b_deg'], bins=bins, alpha=0.75, color='orangered',
        label=f'CSC2-matched (n={len(df_csc2)})')
ax.set_xlabel('|b| (deg)')
ax.set_ylabel('source count')
ax.set_title('Galactic-latitude distribution')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\nBin summary (using |b| bins from src/galactic.py):')
from galactic import bin_counts
print('  no-CSC2:')
print(bin_counts(df).to_string())
print('  CSC2-matched:')
print(bin_counts(df_csc2).to_string())


In [ ]:
# ── Multi-wavelength radio flux + spectral indices ──
# Units folded into column names; display in mJy (Jy ×1000).
RADIO = {
    'NVSS'    : ('NVSS_flux_mJy',     1.0,    1.4),
    'RACS-mid': ('RACS-mid_flux_mJy', 1.0,    1.367),
    'VLASS'   : ('VLASS_flux_Jy',     1000.0, 3.0),
    'AT20G'   : ('AT20G_flux_Jy',     1000.0, 20.0),
}
df['has_any_radio'] = df[['AT20G_match','NVSS_match','RACS-mid_match',
                          'VLASS_match']].fillna(False).any(axis=1)
df_radio = df[df['has_any_radio']].copy()
print(f'{len(df_radio)} / {len(df)} no-CSC2 sources have ≥1 radio match.\n')

print('Radio flux table (mJy):')
hdr = f"  {'id':30s} " + ' '.join(f'{c:>10s}' for c in RADIO)
print(hdr); print('  ' + '-'*(len(hdr)-2))
for _, r in df_radio.head(30).iterrows():
    vals = []
    for cat, (col, scale, nu) in RADIO.items():
        v = r.get(col)
        vals.append(f'{v*scale:>10.1f}' if pd.notna(v) else f'{"—":>10s}')
    print(f"  {r['id']:30s} " + ' '.join(vals))
if len(df_radio) > 30:
    print(f'  ... ({len(df_radio)-30} more omitted)')

# α(1.4→3.0 GHz): NVSS vs VLASS (both mJy after scaling), S_ν ∝ ν^α
both = df_radio.dropna(subset=['NVSS_flux_mJy','VLASS_flux_Jy'])
both = both[(both['NVSS_flux_mJy'] > 0) & (both['VLASS_flux_Jy'] > 0)].copy()
both['VLASS_mJy'] = both['VLASS_flux_Jy'] * 1000.0
alpha = np.log(both['VLASS_mJy'] / both['NVSS_flux_mJy']) / np.log(3.0/1.4)
print(f'\nSpectral index α(1.4→3.0 GHz), n={len(both)}:')
if len(both):
    out = pd.DataFrame({'id': both['id'].values,
                        '|SNR|': both['snr_max'].abs().values,
                        'NVSS_mJy': both['NVSS_flux_mJy'].values,
                        'VLASS_mJy': both['VLASS_mJy'].values,
                        'alpha': alpha.values})
    print(out.to_string(index=False))
    print(f'\n  α median = {np.median(alpha):+.2f}   '
          f'steep(<-0.5): {(alpha<-0.5).sum()}  flat(≥-0.5): {(alpha>=-0.5).sum()}')


In [ ]:
# ── AllWISE W1-W2 color distribution + Stern+2012 caveats ──
df_wise = df.dropna(subset=['AllWISE_flux_mag','AllWISE_W2mag']).copy()
df_wise['W1_W2'] = df_wise['AllWISE_flux_mag'] - df_wise['AllWISE_W2mag']
print(f'{len(df_wise)} / {len(df)} sources have AllWISE W1 and W2 magnitudes.\n')

print(f"W1-W2 distribution:")
print(df_wise['W1_W2'].describe().to_string())
n_stern = int((df_wise['W1_W2'] >= 0.8).sum())
print(f'\nSources with W1-W2 ≥ 0.8 (Stern+2012 AGN-candidate cut, naive): {n_stern}')

print('\nThese candidates (top 10 by W1-W2):')
candidates = df_wise[df_wise['W1_W2'] >= 0.8].nlargest(10, 'W1_W2')
cols = ['id', 'snr_max', 'abs_b_deg', 'AllWISE_flux_mag', 'AllWISE_W2mag', 'W1_W2',
        'AllWISE_qph', 'AllWISE_ccf', 'simbad_otype']
print(candidates[[c for c in cols if c in candidates.columns]].to_string(index=False))

print('\nCAVEAT: Stern+2012 requires Vega mags + WISE source photometry + ')
print('explicit depth/quality cuts. The AllWISE catalog values used here have')
print('NOT been filtered on qph (photometric quality) or ccf (contamination).')
print('In the Galactic plane, where source crowding and IR cirrus are high,')
print('many of these "candidates" are likely chance alignments or contaminated.')
print('Without CSC2 confirmation (this entire subset has none), the W1-W2 cut')
print('alone is NOT sufficient evidence for AGN classification.')

# Quick W1-W2 vs |b| plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(df_wise['abs_b_deg'], df_wise['W1_W2'], s=20, alpha=0.6,
           color='steelblue', label=f'no-CSC2 (n={len(df_wise)})')
ax.axhline(0.8, color='red', linestyle='--', alpha=0.6,
           label='Stern+2012 cut (W1-W2 ≥ 0.8)')
ax.set_xlabel('|b| (deg)'); ax.set_ylabel('W1 − W2  (AllWISE catalog, mag)')
ax.set_title('AllWISE W1-W2 vs Galactic latitude (no-CSC2 subset)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Save no-CSC2 subset CSV ──
KEEP_PREFIX = ('id','snr_max','abs_b_deg','l_deg','b_deg',
               'sigma_pos_arcsec_heuristic',
               'simbad_','CSC2_','AT20G_','NVSS_','RACS-mid_','VLASS_','AllWISE_',
               'has_csc2_match_within_3sigma','csc2_',
               'counterpart_fingerprint','has_any_radio','has_ir','has_simbad')
keep_cols = [c for c in df.columns if c.startswith(KEEP_PREFIX)]
out = df[keep_cols].copy()
out_path = os.path.join(OUT, 'v4_no_csc2_subset_summary.csv')
out.to_csv(out_path, index=False)
print(f'Saved {len(out)} rows × {len(out.columns)} cols → {out_path}')


In [ ]:
# ── Per-source cutout panels: one source ──
# Same method as in analyze_v4_csc2_candidates.ipynb: src/source_figure.plot_source_by_name
# Reads data/fits_cache/ (run scripts/precache_fits.py first).
from source_figure import plot_source_by_name

fig, failures = plot_source_by_name(df['id'].iloc[0], fov=45)
plt.show()
print(failures or 'all panels drawn')

In [ ]:
# ── Per-source cutout panels: top N by |SNR| ──
# All 73 sources as PNGs: python scripts/plot_source_panels.py --all
N_SHOW = 5
for name in df['id'].head(N_SHOW):
    fig, failures = plot_source_by_name(name, fov=45)
    plt.show()
    for key, err in failures.items():
        print(f'  [{key}] {err[:90]}')